In [1]:
#PROJEKT UCZENIE MASZYNOWE



In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import tree
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold
import sklearn.model_selection as skm
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import balanced_accuracy_score

In [3]:
X_train = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_train_data.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_train_labels.csv")

In [4]:
X_test = pd.read_csv("https://raw.githubusercontent.com/kozaka93/2025Z-MachineLearning/refs/heads/main/project/artifical_test_data.csv")

In [5]:
print("---WYMIARY---")
print("X_train:", X_train.shape)
print("y_train:", np.shape(y_train))
print("X_test :", X_test.shape)

print("---ZGODNOSC STRUKTUR DANYCH---")
assert X_train.shape[0] == len(y_train), "X_train i y_train mają różną liczbę wierszy!"
assert X_train.shape[1] == X_test.shape[1], "X_train i X_test mają różną liczbę kolumn!"
same_cols = list(X_train.columns) == list(X_test.columns)
print("Same column order in train/test?:", same_cols)

print("---PROPORCJE---")
y_unique, y_counts = np.unique(y_train, return_counts=True)
print("Unique labels:", y_unique)
print("Counts:", dict(zip(y_unique, y_counts)))
print("Proportions:", {k: round(v/len(y_train), 4) for k, v in dict(zip(y_unique, y_counts)).items()})
print("Any NaN in y_train?:", np.isnan(y_train).any() if np.issubdtype(np.array(y_train).dtype, np.number) else "n/a")

print("---BRAKI---")
nan_any = X_train.isna().any().any()
print("Any NaN in X_train?:", nan_any)
if nan_any:
    nan_cols = X_train.isna().sum()
    nan_cols = nan_cols[nan_cols > 0].sort_values(ascending=False)
    print("NaN columns count:", len(nan_cols))
    print("Top 15 NaN columns:\n", nan_cols.head(15).to_string())

print("---TYPY DANYCH---")
non_numeric = [c for c in X_train.columns if not pd.api.types.is_numeric_dtype(X_train[c])]
print("Non-numeric columns:", non_numeric)
print("Dtype summary:\n", X_train.dtypes.value_counts().to_string())

print("---STALE KOLUMNY---czy sa jakies stale kolumny (ktore nic nie wnosza i mozna sie ich od razu pozbyc)")
# stałe kolumny (wariancja = 0)
const_cols = [c for c in X_train.columns if X_train[c].nunique(dropna=False) <= 1]
print("Constant columns:", len(const_cols))
if const_cols:
    print("Examples:", const_cols[:20])

print("---DUPLIKATY---")
dup_train = X_train.duplicated().sum()
dup_test = X_test.duplicated().sum()
print("Duplicate rows in X_train:", dup_train)
print("Duplicate rows in X_test :", dup_test)


print("---STATYSTYKI---pierwszy rzut oka na statystyki danych")
desc = X_train.select_dtypes(include=[np.number]).describe().T
print(desc[["mean", "std", "min", "max"]].head(10).to_string())
#wydaje sie, ze wszystko jest w normie

---WYMIARY---
X_train: (1500, 100)
y_train: (1500, 1)
X_test : (500, 100)
---ZGODNOSC STRUKTUR DANYCH---
Same column order in train/test?: True
---PROPORCJE---
Unique labels: [1 2]
Counts: {1: 755, 2: 745}
Proportions: {1: 0.5033, 2: 0.4967}
Any NaN in y_train?: Class    False
dtype: bool
---BRAKI---
Any NaN in X_train?: False
---TYPY DANYCH---
Non-numeric columns: []
Dtype summary:
 int64    100
---STALE KOLUMNY---czy sa jakies stale kolumny (ktore nic nie wnosza i mozna sie ich od razu pozbyc)
Constant columns: 0
---DUPLIKATY---
Duplicate rows in X_train: 0
Duplicate rows in X_test : 0
---STATYSTYKI---pierwszy rzut oka na statystyki danych
           mean         std    min    max
V1   478.896667   26.423515  379.0  563.0
V2   479.015333   29.103558  361.0  571.0
V3   487.754000   73.378083  282.0  677.0
V4   478.756667   33.332599  366.0  593.0
V5   482.687333   13.858967  435.0  535.0
V6   478.743333   22.437790  394.0  553.0
V7   481.036667   22.502701  409.0  566.0
V8   481.80600

In [6]:
#Będę BUDOWAĆ DWA PIPELINY A0 ORAZ A1.
#Pipeline A0 pełni rolę punktu odniesienia, natomiast pipeline A1 wprowadza selekcję cech i bardziej złożony model w celu poprawy jakości predykcji.

#CZYLI na poziomie A0 mierzymy jakość po to, aby wiedzieć, czy i w jakim kierunku warto zwiększać złożoność modelu w A1.

if isinstance(y_train, pd.DataFrame):
    y_train = y_train.iloc[:, 0].values
elif isinstance(y_train, pd.Series):
    y_train = y_train.values
    
A0 = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),  #jeśli nie ma NaN, nic nie psuje
    ("var", VarianceThreshold()),                   #usuwa cechy stałe (wariancja = 0)
    ("scaler", StandardScaler()),                   #ujednolica skalę
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=5000,
        random_state=42
    ))
])

#Oceniam jakość A0 licząc miarę BA przy dodatkowym użyciu kroswalidacji
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(A0, X_train, y_train, cv=cv, scoring="balanced_accuracy", n_jobs=-1)

print("A0 balanced accuracy (5-fold):", np.round(scores, 4))
print("A0 mean:", scores.mean().round(4), "| std:", scores.std().round(4))

A0 balanced accuracy (5-fold): [0.5837 0.55   0.5799 0.5867 0.583 ]
A0 mean: 0.5767 | std: 0.0135


In [7]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
#PREPROCESSING
#Wszystkie etapy przetwarzania danych zostały włączone do pipeline’u, aby uniknąć przecieku informacji i zapewnić poprawną walidację.
#Poprawne przygotowanie danych usuwa problemy techniczne i szum

rf_selector = SelectFromModel(
    estimator=RandomForestClassifier(
        n_estimators=800,
        random_state=42,
        n_jobs=-1
    ),
    #przy czym mamy ograniczenie, mówiące ile najważniejszych cech model może zostawić
    threshold=-np.inf,
    max_features=20   #20 istotnie jest potwierdzone jako najlepsze ograniczenie w dalszych eksperymentach.
)

#wspólna "taśma":porządki takie jak skalowanie, usuwanie stałych cech (lub o małej wariancji) oraz zastosowanie selekcji zmiennych.
#scaling nie szkodzi drzewom, a jest potrzebny dla LR.
common_prefix = [
    ("imputer", SimpleImputer(strategy="median")),
    ("var", VarianceThreshold()),  
    ("select", rf_selector),
    ("scaler", StandardScaler())
]

# --- A1.1: RF selekcja + prosty model końcowy (LR) ---
pipe_A11 = Pipeline(steps=common_prefix + [
    ("model", LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=5000,
        random_state=42
    ))
])

# --- A1.2: RF selekcja + RF jako model końcowy ---
pipe_A12 = Pipeline(steps=common_prefix + [
    ("model", RandomForestClassifier(
        n_estimators=1000,
        random_state=42,
        n_jobs=-1
    ))
])

# --- A1.3: RF selekcja + boosting jako model końcowy (szybki i solidny) ---
pipe_A13 = Pipeline(steps=common_prefix + [
    ("model", HistGradientBoostingClassifier(
        random_state=42,
        max_depth=3,
        learning_rate=0.05,
        max_iter=400
    ))
])

pipelines = {
    "A1.1 RF-select + LR": pipe_A11,
    "A1.2 RF-select + RF": pipe_A12,
    "A1.3 RF-select + Boosting": pipe_A13
}

results = []
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="balanced_accuracy", n_jobs=-1)
    results.append((name, scores.mean(), scores.std(), scores))

#Wyniki w tabeli
df_res = pd.DataFrame(
    [(n, m, s) for n, m, s, _ in results],
    columns=["Model", "Mean BA", "Std BA"]
).sort_values("Mean BA", ascending=False)

print(df_res.to_string(index=False))

#pełne wyniki foldów (dla przejrzystości)
for n, m, s, sc in sorted(results, key=lambda x: x[1], reverse=True):
    print(f"\n{n}")
    print("folds:", np.round(sc, 4), "| mean:", round(m, 4), "| std:", round(s, 4))

#WYBÓR NAJLEOSZEGO WARIANTU
best_name, best_mean, best_std, _ = sorted(results, key=lambda x: x[1], reverse=True)[0]
best_pipe = pipelines[best_name]
print("\nBEST:", best_name, "| mean:", round(best_mean, 4), "| std:", round(best_std, 4))

                    Model  Mean BA   Std BA
      A1.2 RF-select + RF 0.870696 0.026911
A1.3 RF-select + Boosting 0.816779 0.034086
      A1.1 RF-select + LR 0.598622 0.024590

A1.2 RF-select + RF
folds: [0.8801 0.8736 0.9    0.8199 0.8798] | mean: 0.8707 | std: 0.0269

A1.3 RF-select + Boosting
folds: [0.8602 0.7637 0.8334 0.7933 0.8334] | mean: 0.8168 | std: 0.0341

A1.1 RF-select + LR
folds: [0.5633 0.6103 0.5834 0.6    0.6361] | mean: 0.5986 | std: 0.0246

BEST: A1.2 RF-select + RF | mean: 0.8707 | std: 0.0269


In [8]:
#badam ILE NAJISTOTNIEJSZYCH CECH ZOSTAWIC na podstawie wybranego modeku A1.2 . Skupiam się tylko na nim,
#ponieważ różnice między miarą BA różnych modeli były spore,
#zatem raczej nie są wynikiem przypadku.



In [9]:
K_list = [5, 10, 20, 30, 40, 60, 100]

rows = []
for K in K_list:
    rf_selector_K = SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ),
        threshold=-np.inf,
        max_features=K
    )

    common_prefix_K = [
        ("imputer", SimpleImputer(strategy="median")),
        ("var", VarianceThreshold()),
        ("select", rf_selector_K),
        ("scaler", StandardScaler())
    ]

    pipe = Pipeline(steps=common_prefix_K + [
        ("model", RandomForestClassifier(
            n_estimators=800,
            random_state=42,
            n_jobs=-1
        ))
    ])

    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="balanced_accuracy", n_jobs=-1)
    rows.append((K, scores.mean(), scores.std(), scores))

dfK = pd.DataFrame([(k, m, s) for k, m, s, _ in rows], columns=["K", "Mean BA", "Std BA"])\
        .sort_values("Mean BA", ascending=False)

print(dfK.to_string(index=False))

for k, m, s, sc in sorted(rows, key=lambda x: x[1], reverse=True):
    print(f"\nK={k}")
    print("folds:", np.round(sc, 4), "| mean:", round(m, 4), "| std:", round(s, 4))

best_K = sorted(rows, key=lambda x: x[1], reverse=True)[0][0]
print("\nBEST K:", best_K)

  K  Mean BA   Std BA
 20 0.870714 0.025467
 30 0.861434 0.026555
 10 0.845478 0.037898
 40 0.841513 0.030249
 60 0.818219 0.029557
100 0.800338 0.031681
  5 0.786773 0.027958

K=20
folds: [0.8768 0.8736 0.9001 0.8233 0.8798] | mean: 0.8707 | std: 0.0255

K=30
folds: [0.8802 0.8504 0.8934 0.8166 0.8666] | mean: 0.8614 | std: 0.0266

K=10
folds: [0.8136 0.8204 0.8934 0.81   0.89  ] | mean: 0.8455 | std: 0.0379

K=40
folds: [0.8536 0.8171 0.8868 0.8001 0.8499] | mean: 0.8415 | std: 0.0302

K=60
folds: [0.8336 0.7939 0.8601 0.7769 0.8267] | mean: 0.8182 | std: 0.0296

K=100
folds: [0.8204 0.764  0.8436 0.7638 0.8099] | mean: 0.8003 | std: 0.0317

K=5
folds: [0.7701 0.7836 0.7967 0.75   0.8334] | mean: 0.7868 | std: 0.028

BEST K: 20


In [10]:
#zatem początkowo dobrana liczba cech K=20 daje nam najlepszy model, o najwyższym średnim BA i niewielkim odchyleniu tej miary.

In [11]:
#DOBOR HIPERPARAMETRÓW. W dalszym ciągu zajmujemy się dostrajaniem modelu z doborem cech RandomForest oraz RF jako głównym modelem.

In [12]:
param_grid = {
    "model__max_depth": [None, 8, 12],
    "model__min_samples_leaf": [1, 2, 5],
}

grid = GridSearchCV(
    estimator=best_pipe,  
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)
print("Best balanced accuracy (CV):", grid.best_score_)
print("Best parameters:")
for k, v in grid.best_params_.items():
    print(f"  {k}: {v}")

best_pipe_tuned = grid.best_estimator_

Best balanced accuracy (CV): 0.8706964753989066
Best parameters:
  model__max_depth: None
  model__min_samples_leaf: 1


In [13]:
best_pipe_tuned.fit(X_train, y_train)

proba_test = best_pipe_tuned.predict_proba(X_test)

classes = best_pipe_tuned.named_steps["model"].classes_
idx_class1 = int(np.where(classes == 1)[0][0])

p_class2 = proba_test[:, idx_class1]

np.savetxt("333047_artifical_prediction.txt", p_class2, fmt="%.10f")